In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import

In [2]:
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 82.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 87.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 38.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 99.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.6 MB/s eta 0:00:00
  Created wheel for sentence_transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=cfc9d96ea22c7073d88e739ab64f5ea0a4f5d31bbcd462ff30029f2b15f1808d
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence_transformers


In [3]:
import re
import pandas as pd
import numpy as np
import random
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

## Random Seed

In [4]:
SEED = 0

np.random.seed(SEED)
random.seed(SEED)

## Load Data

In [5]:
df = pd.read_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/news.csv')
df.head()

,id,title,contents
0,NEWS_00000,Spanish coach facing action in race row,MADRID (AFP) - Spanish national team coach Lui...
1,NEWS_00001,Bruce Lee statue for divided city,"In Bosnia, where one man #39;s hero is often a..."
2,NEWS_00002,Only Lovers Left Alive's Tilda Swinton Talks A...,Yasmine Hamdan performs 'Hal' which she also s...
3,NEWS_00003,Macromedia contributes to eBay Stores,Macromedia has announced a special version of ...
4,NEWS_00004,Qualcomm plans to phone it in on cellular repairs,Over-the-air fixes for cell phones comes to Qu...


In [6]:
# 제목 + 내용
df['text'] = df['title'] + ' : ' + df['contents']
df.head()

,id,title,contents,text
0,NEWS_00000,Spanish coach facing action in race row,MADRID (AFP) - Spanish national team coach Lui...,Spanish coach facing action in race row : MADR...
1,NEWS_00001,Bruce Lee statue for divided city,"In Bosnia, where one man #39;s hero is often a...","Bruce Lee statue for divided city : In Bosnia,..."
2,NEWS_00002,Only Lovers Left Alive's Tilda Swinton Talks A...,Yasmine Hamdan performs 'Hal' which she also s...,Only Lovers Left Alive's Tilda Swinton Talks A...
3,NEWS_00003,Macromedia contributes to eBay Stores,Macromedia has announced a special version of ...,Macromedia contributes to eBay Stores : Macrom...
4,NEWS_00004,Qualcomm plans to phone it in on cellular repairs,Over-the-air fixes for cell phones comes to Qu...,Qualcomm plans to phone it in on cellular repa...


## Pre-processing

In [7]:
def preprocess_text(text):
    # URL 제거
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 해시태그 제거
    text = re.sub(r'#\w+', '', text)

    # 멘션 제거
    text = re.sub(r'@\w+', '', text)

    # 이모지 제거
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 공백 및 특수문자 제거
    text = re.sub(r'\s+', ' ', text).strip()

    # 숫자 제거
    text = re.sub(r'\d+', '', text)

    return text.lower()

In [8]:
df['processed_text'] = df['text'].apply(preprocess_text)

## Feature Extraction

In [10]:
# Sentence BERT 모델 로드
model = SentenceTransformer('paraphrase-MiniLM-L12-v2')

# 텍스트 feature 추출
sentence_embeddings = model.encode(df['text'].tolist())

# 추출한 feature를 데이터프레임에 저장
df_embeddings = pd.DataFrame(sentence_embeddings)

## Clustering

In [11]:
# Sentence BERT 임베딩을 사용하여 군집화 수행
kmeans = KMeans(n_clusters=6, random_state=SEED)

df['kmeans_cluster'] = kmeans.fit_predict(sentence_embeddings)

## Post-processing

### Tech: 0 -> 4

In [12]:
df[df['kmeans_cluster'] == 0]['text']

3        Macromedia contributes to eBay Stores : Macrom...
4        Qualcomm plans to phone it in on cellular repa...
5        Thomson to Back Both Blu-ray and HD-DVD : Comp...
23       FTC Files First Lawsuit Against Spyware Concer...
31       Sony PSP Draws Crowds and Lines on First Day (...
                               ...                        
59972    SBC Details Fiber Plans : SBC Communications I...
59976    The Sims Go to College : November 22, 2004 - I...
59978    SMIC to challenge latest TSMC infringement cla...
59983    Partners Weigh In On Firefox, IE Faceoff : Sol...
59991    Irish talk of softening schedule a little : wh...
Name: text, Length: 9504, dtype: object

### Business: 1 -> 0

In [15]:
df[df['kmeans_cluster'] == 1]['text']

7        Bump Stock Maker Resumes Sales One Month After...
20       Deere's Color Is Green : With big tractors, bi...
27       Kmart-Sears merger about price, quality : Aver...
36       Agencies Postpone Issuing New Rules Until Afte...
39       Ricky Williams ordered to repay Dolphins : For...
                               ...                        
59985    European Shares Tread Water (Reuters) : Reuter...
59986    Higher trade growth predicted in 2004 despite ...
59994    Chain Store Sales Rise in the Latest Week : NE...
59996    After Steep Drop, Price of Oil Rises : The fre...
59998    Albertsons on the Rebound : The No. 2 grocer r...
Name: text, Length: 9980, dtype: object

### Sports: 2 -> 3

In [16]:
df[df['kmeans_cluster'] == 2]['text']

0        Spanish coach facing action in race row : MADR...
6        Time to Talk Baseball : It's time to talk abou...
13       GAME DAY PREVIEW Game time: 6:00 PM : CHARLOTT...
21       Blake Leeper Wants to Be the First American Pa...
22       College Basketball: Georgia Tech, UConn Win : ...
                               ...                        
59989    Wigley makes clean break from Southampton : Wi...
59990    Wizards Down Galaxy : Kansas City secures the ...
59993    Olson: Hoping to preserve the joy of Sox : Sin...
59995    Dolphins Break Through, Rip Rams For First Win...
59997    Pro football: Culpepper puts on a show : To sa...
Name: text, Length: 12029, dtype: object

### World: 3 -> 5

In [17]:
df[df['kmeans_cluster'] == 3]['text']

1        Bruce Lee statue for divided city : In Bosnia,...
10       Harry #39;s argy-bargy : PRINCE Charles has as...
29       Israel Kills 3 Palestinians in Big Gaza Incurs...
34       The Folly of the Sole Superpower Writ Small au...
37       Deep Impact Space Probe Aims to Slam Into Come...
                               ...                        
59982    Militants kill 12 in J amp;K ahead of PMs visi...
59984    The Plot Thickens: Testing European Tolerance ...
59987    Nepal Seeks to Break Rebel Siege with Air Patr...
59992    US troops on offensive in Iraq : US troops wen...
59999    Cassini Craft Spies Saturn Moon Dione (AP) : A...
Name: text, Length: 11579, dtype: object

### Entertainment: 4 -> 1

In [18]:
df[df['kmeans_cluster'] == 4]['text']

2        Only Lovers Left Alive's Tilda Swinton Talks A...
8        Obama Marks Anniversary Of 9/11 Attacks With M...
9        Republican Congressman Says Trump Should Apolo...
11       Kerry rolls out tax-cut plan for middle class ...
12       Read Live Updates From The South Carolina Demo...
                               ...                        
59953    A Dozen Lessons Donald Trump Could Learn From ...
59954    You Don't Have To Agree With Donald Trump To B...
59966    Mitt Romney Lambasts Donald Trump As A 'Phony'...
59969    'American Hustle' '12 Years a Slave' Lead 2014...
59981    Khloe Kardashian Gets A Hilarious Lesson In Po...
Name: text, Length: 13858, dtype: object

### Politics: 5 -> 2

In [22]:
df[df['kmeans_cluster'] == 5]['text']

18       A Fair Way to Choose Candidates for Republican...
25       Be on TOP : //www.huffingtonpost.com/entry/be-...
33       Memo To EPA Chief Pruitt : //www.huffingtonpos...
68       Satire Will Not Save Us : //www.huffingtonpost...
76       WATCH : //www.huffingtonpost.com/entry/perrish...
                               ...                        
59903    'Prometheus' Trailer 2 : //www.huffingtonpost....
59906                Report : //www.huffingtonpost.comhttp
59912    Joe Biden : //www.huffingtonpost.com/entry/joe...
59923    Bernie in Bush-Obama America : //www.huffingto...
59988    Obama To Call For More Icebreakers In The Arct...
Name: text, Length: 3050, dtype: object

### Mapping

In [23]:
mapping_dict = {
    0: 4,
    1: 0,
    2: 3,
    3: 5,
    4: 1,
    5: 2
}

In [24]:
df['mapping'] = df['kmeans_cluster'].apply(lambda x: mapping_dict[x])

## Submission

In [25]:
sample = pd.read_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/sample_submission.csv')

In [26]:
sample['category'] = df['mapping'].values
sample['category'].head()

0    3
1    5
2    1
3    4
4    4
Name: category, dtype: int64

In [27]:
sample.to_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/paraphrase-MiniLM-L12-v2 .csv', index=False)